# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[-0.29510423  0.1068003  -0.43523171 -0.69415258  0.02661787]
 [ 0.55093182  0.43291438 -0.66035325 -0.06429803  0.11879739]
 [-0.76834155  0.63006007 -0.3286156   0.81764852 -0.96857704]
 [-0.06774331  0.8558794  -0.30714485 -0.980672   -0.62003066]
 [-0.433614   -0.99960589 -0.14863463 -0.84700138  0.3058156 ]
 [-0.68251017 -0.40540145 -0.24761987 -0.51004182  0.748183  ]
 [-0.77390573 -0.67027872  0.52795189  0.24742719 -0.0752349 ]
 [-0.39993445  0.27510234  0.86712322  0.45277544 -0.00977359]
 [-0.92392614 -0.72671577 -0.51134801 -0.99505028  0.9767449 ]
 [-0.6603465  -0.6826477  -0.94683999 -0.06060249 -0.8213706 ]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a1', 'a1', 'a2', 'a2', 'a1', 'a1', 'a2', 'a2', 'a2', 'a2']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [0, 0, 0, 1, 0, 1, 1, 0, 0, 1]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:34,  1.03s/it]

SVI:   3%|▎         | 1/34 [00:01<00:34,  1.03s/it, loss=2692.9753]

SVI:   6%|▌         | 2/34 [00:01<00:33,  1.03s/it, loss=2305.9502]

SVI:   9%|▉         | 3/34 [00:01<00:31,  1.03s/it, loss=2178.6926]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.10it/s, loss=3229.6448]

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.10it/s, loss=2790.1875]

SVI:  97%|█████████▋| 33/34 [00:00<00:00,  1.10it/s, loss=2245.9695]

SVI: 100%|██████████| 34/34 [00:01<00:00, 22.26it/s, loss=2245.9695]

SVI: 100%|██████████| 34/34 [00:01<00:00, 22.26it/s, loss=4723.2251]